# Build Sign Count Dataset

This notebook converts the merged long TCAV CSV into a sample-level wide dataset using `sign_count` features only.


In [1]:
from pathlib import Path

import pandas as pd


In [2]:
# ===== Config =====
PROJECT_ROOT = Path('/home/SpeakerRec/BioVoice')
TCAV_ROOT = PROJECT_ROOT / 'redimnet' / 'tcav' / 'captum_tcav' / 'asvspoof5'
OUTPUT_SUBDIR = 'subset_20spk_20utts_per_spk_one_logistic_head'
OUTPUT_DIR = TCAV_ROOT / 'output' / OUTPUT_SUBDIR

INPUT_CSV = OUTPUT_DIR / 'merged_tcav_long.csv'
OUTPUT_CSV = OUTPUT_DIR / 'merged_tcav_sign_count_wide.csv'

# Set True if you want to exclude A12 from the concept-level dataset.
EXCLUDE_A12 = False

print('INPUT_CSV =', INPUT_CSV)
print('Exists =', INPUT_CSV.exists())


INPUT_CSV = /home/SpeakerRec/BioVoice/redimnet/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_long.csv
Exists = True


In [3]:
df = pd.read_csv(INPUT_CSV)
print('Loaded rows =', len(df))
display(df.head())


Loaded rows = 153600


,system_id,source_partition,split,speaker_id,utt_id,magnitude,sign_count,concept_name,target_class,source_file
0,A09,dev,test,D_0430,D_0000036016,0.258070,1.0,long_constant_thick,bonafide,tcav_dev_A09_bonafide.csv
1,A09,dev,test,D_0430,D_0000036016,-0.032694,0.0,long_dropping_flat_thick,bonafide,tcav_dev_A09_bonafide.csv
2,A09,dev,test,D_0430,D_0000036016,-0.226185,0.0,long_dropping_steep_thick,bonafide,tcav_dev_A09_bonafide.csv
3,A09,dev,test,D_0430,D_0000036016,0.226079,1.0,long_dropping_steep_thin,bonafide,tcav_dev_A09_bonafide.csv
4,A09,dev,test,D_0430,D_0000036016,0.132804,1.0,long_rising_flat_thick,bonafide,tcav_dev_A09_bonafide.csv


In [4]:
required_cols = {
    'utt_id',
    'speaker_id',
    'split',
    'source_partition',
    'system_id',
    'target_class',
    'concept_name',
    'sign_count',
}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing columns: {sorted(missing)}')

if EXCLUDE_A12:
    df = df[df['system_id'] != 'A12'].copy()

df['binary_label'] = (df['target_class'] == 'spoof').astype(int)

print('Rows after filtering =', len(df))
print('Unique utterances =', df['utt_id'].nunique())
print('Systems =', sorted(df['system_id'].unique().tolist()))
print('Concepts =', sorted(df['concept_name'].unique().tolist()))


Rows after filtering = 153600
Unique utterances = 8307
Systems = ['A01', 'A02', 'A03', 'A04', 'A05', 'A06', 'A07', 'A08', 'A09', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']
Concepts = ['long_constant_thick', 'long_dropping_flat_thick', 'long_dropping_steep_thick', 'long_dropping_steep_thin', 'long_rising_flat_thick', 'long_rising_steep_thick', 'long_rising_steep_thin', 'short_constant_thick', 'short_dropping_steep_thick', 'short_dropping_steep_thin', 'short_rising_steep_thick', 'short_rising_steep_thin']


In [5]:
meta_cols = ['utt_id', 'speaker_id', 'split', 'source_partition', 'system_id', 'target_class', 'binary_label']

wide_df = (
    df.pivot_table(
        index=meta_cols,
        columns='concept_name',
        values='sign_count',
        aggfunc='first',
    )
    .reset_index()
)

wide_df.columns.name = None
concept_cols = [c for c in wide_df.columns if c not in meta_cols]
wide_df = wide_df.sort_values(['source_partition', 'system_id', 'target_class', 'speaker_id', 'utt_id']).reset_index(drop=True)

print('Wide rows =', len(wide_df))
print('Feature columns =', len(concept_cols))
print('Missing feature values =', int(wide_df[concept_cols].isna().sum().sum()))
display(wide_df.head())


Wide rows = 12800
Feature columns = 12
Missing feature values = 0


,utt_id,speaker_id,split,source_partition,system_id,target_class,binary_label,long_constant_thick,long_dropping_flat_thick,long_dropping_steep_thick,long_dropping_steep_thin,long_rising_flat_thick,long_rising_steep_thick,long_rising_steep_thin,short_constant_thick,short_dropping_steep_thick,short_dropping_steep_thin,short_rising_steep_thick,short_rising_steep_thin
0,D_0000036016,D_0430,test,dev,A09,bonafide,0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,D_0000207208,D_0430,test,dev,A09,bonafide,0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0
2,D_0000313804,D_0430,test,dev,A09,bonafide,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,D_0000437998,D_0430,test,dev,A09,bonafide,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,D_0000438019,D_0430,test,dev,A09,bonafide,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [6]:
summary_df = pd.DataFrame({
    'rows': [len(wide_df)],
    'utterances': [wide_df['utt_id'].nunique()],
    'speakers': [wide_df['speaker_id'].nunique()],
    'systems': [wide_df['system_id'].nunique()],
    'sign_count_features': [len(concept_cols)],
    'spoof_rows': [int((wide_df['binary_label'] == 1).sum())],
    'bonafide_rows': [int((wide_df['binary_label'] == 0).sum())],
})
display(summary_df)

display(
    wide_df.groupby(['split', 'target_class'])['utt_id']
    .nunique()
    .rename('n_utterances')
    .reset_index()
    .sort_values(['split', 'target_class'])
)


,rows,utterances,speakers,systems,sign_count_features,spoof_rows,bonafide_rows
0,12800,8307,81,16,12,6400,6400


,split,target_class,n_utterances
0,test,bonafide,1907
1,test,spoof,6400


In [7]:
wide_df.to_csv(OUTPUT_CSV, index=False)
print('Saved sign-count wide dataset to:', OUTPUT_CSV)


Saved sign-count wide dataset to: /home/SpeakerRec/BioVoice/redimnet/tcav/captum_tcav/asvspoof5/output/subset_20spk_20utts_per_spk_one_logistic_head/merged_tcav_sign_count_wide.csv
